In [1]:
import yfinance as yf


In [11]:
ticker = "AAPL"
data = yf.download(ticker, start="2020-01-01", end="2025-01-01", interval="1d")

print(data.head())

"""
open : 시가
high : 고가
low : 저가
close : 종가
adj close : 수정 종가(주식 분할, 배당 반영)
volume : 거래량



"""



/var/folders/57/ff11tjnn7k3d1d5693_01dg00000gn/T/ipykernel_5771/2949116673.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start="2020-01-01", end="2025-01-01", interval="1d")
[*********************100%***********************]  1 of 1 completed

Price           Close       High        Low       Open     Volume
Ticker           AAPL       AAPL       AAPL       AAPL       AAPL
Date                                                             
2020-01-02  72.620842  72.681289  71.373218  71.627092  135480400
2020-01-03  71.914810  72.676439  71.689950  71.847110  146322800
2020-01-06  72.487854  72.526541  70.783256  71.034717  118387200
2020-01-07  72.146935  72.753816  71.926907  72.497522  108872000
2020-01-08  73.307495  73.609729  71.849518  71.849518  132079200


'\nopen : 시가\nhigh : 고가\nlow : 저가\nclose : 종가\nadj close : 수정 종가(주식 분할, 배당 반영)\nvolume : 거래량\n\n\n\n'

In [12]:
data = data.copy()
data['Stock growth rate(%)'] = data['Close'].pct_change() * 100
data['Stock growth rate(%)'] = data['Stock growth rate(%)'].round(2)
print(data.head())

Price           Close       High        Low       Open     Volume  \
Ticker           AAPL       AAPL       AAPL       AAPL       AAPL   
Date                                                                
2020-01-02  72.620842  72.681289  71.373218  71.627092  135480400   
2020-01-03  71.914810  72.676439  71.689950  71.847110  146322800   
2020-01-06  72.487854  72.526541  70.783256  71.034717  118387200   
2020-01-07  72.146935  72.753816  71.926907  72.497522  108872000   
2020-01-08  73.307495  73.609729  71.849518  71.849518  132079200   

Price      Stock growth rate(%)  
Ticker                           
Date                             
2020-01-02                  NaN  
2020-01-03                -0.97  
2020-01-06                 0.80  
2020-01-07                -0.47  
2020-01-08                 1.61  


In [13]:
# RSI 수식 직접 구현
def compute_rsi(series, window=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

# RSI 컬럼 추가
data['RSI_14'] = compute_rsi(data['Close']).round(2)

# 확인
print(data[['Stock growth rate(%)', 'RSI_14']].head(20))

Price      Stock growth rate(%) RSI_14
Ticker                                
Date                                  
2020-01-02                  NaN    NaN
2020-01-03                -0.97    NaN
2020-01-06                 0.80    NaN
2020-01-07                -0.47    NaN
2020-01-08                 1.61    NaN
2020-01-09                 2.12    NaN
2020-01-10                 0.23    NaN
2020-01-13                 2.14    NaN
2020-01-14                -1.35    NaN
2020-01-15                -0.43    NaN
2020-01-16                 1.25    NaN
2020-01-17                 1.11    NaN
2020-01-21                -0.68    NaN
2020-01-22                 0.36    NaN
2020-01-23                 0.48  71.90
2020-01-24                -0.29  75.40
2020-01-27                -2.94  59.51
2020-01-28                 2.83  67.41
2020-01-29                 2.09  68.47
2020-01-30                -0.14  63.88


In [14]:
def compute_stochastic_oscillator(df, k_window=14, d_window=3):
    # ① N일 동안의 최저가 및 최고가
    low_min = df['Low'].rolling(window=k_window).min()
    high_max = df['High'].rolling(window=k_window).max()

    # ② %K 계산
    percent_k = ((df['Close'] - low_min) / (high_max - low_min)) * 100

    # ③ %D 계산 (%K의 이동평균)
    percent_d = percent_k.rolling(window=d_window).mean()

    return percent_k.round(2), percent_d.round(2)


In [15]:
# 스토캐스틱 지표 계산
data['%K'], data['%D'] = compute_stochastic_oscillator(data)

# 확인
print(data[['Stock growth rate(%)', 'RSI_14', '%K', '%D']].head(20))

Price      Stock growth rate(%) RSI_14     %K     %D
Ticker                                              
Date                                                
2020-01-02                  NaN    NaN    NaN    NaN
2020-01-03                -0.97    NaN    NaN    NaN
2020-01-06                 0.80    NaN    NaN    NaN
2020-01-07                -0.47    NaN    NaN    NaN
2020-01-08                 1.61    NaN    NaN    NaN
2020-01-09                 2.12    NaN    NaN    NaN
2020-01-10                 0.23    NaN    NaN    NaN
2020-01-13                 2.14    NaN    NaN    NaN
2020-01-14                -1.35    NaN    NaN    NaN
2020-01-15                -0.43    NaN    NaN    NaN
2020-01-16                 1.25    NaN    NaN    NaN
2020-01-17                 1.11    NaN    NaN    NaN
2020-01-21                -0.68    NaN    NaN    NaN
2020-01-22                 0.36    NaN  91.59    NaN
2020-01-23                 0.48  71.90  97.21    NaN
2020-01-24                -0.29  75.40  83.58 